# Step 3: Dataset Setup
## Different options
- First one is downloading using a script that places the data in the download folder (usually recommended)
- Second one is uploading the dataset to your personal/institutional Google Drive and load it from there ([Read More](https://saturncloud.io/blog/google-colab-how-to-read-data-from-my-google-drive/))
- Place the download script directly here on colab

You are free to do as you please in this phase.


In [1]:
# Import all required libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from pathlib import Path

# Setup project path
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'ravdess_train' else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

# Print system and environment info
print("="*80)
print("🔧 PROJECT ENVIRONMENT INFO")
print("="*80)
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"Current Working Directory: {os.getcwd()}")
print(f"Project Root: {PROJECT_ROOT}")
print()

# Check CUDA availability
if torch.cuda.is_available():
    print(f"✅ CUDA is AVAILABLE")
    print(f"   GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")
    print(f"   Number of GPUs: {torch.cuda.device_count()}")
    device = torch.device('cuda')
else:
    print(f"❌ CUDA is NOT available - Using CPU")
    device = torch.device('cpu')

print(f"   Default Device: {device}")
print("="*80)

🔧 PROJECT ENVIRONMENT INFO
Python Version: 3.10.0
PyTorch Version: 2.10.0+cu128
NumPy Version: 2.2.6
Current Working Directory: d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25\ravdess_train
Project Root: d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25

✅ CUDA is AVAILABLE
   GPU Device: NVIDIA GeForce RTX 5060 Ti
   CUDA Version: 12.8
   Number of GPUs: 1
   Default Device: cuda


In [2]:
from utils.download_dataset_local import dowload_ravdess_local

dataset_path = dowload_ravdess_local()
if dataset_path:
    print(f"✅ Downloaded RAVDESS dataset locally in {dataset_path}...")
else:
    print("❌ RAVDESS dataset download failed.")
    
ravdess_path = dataset_path


--- Download RAVDESS (locale) ---
✓ RAVDESS già presente: d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25\data\ravdess
Numero di file: 2880
✅ Downloaded RAVDESS dataset locally in d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25\data\ravdess...


In [3]:
from torch.utils.data import DataLoader
from dataset.custom_ravdess_dataset import CustomRAVDESSDataset
from utils.get_dataset_statistics import print_dataset_stats

print("="*80)
print("🔄 CREAZIONE DATASET E DATALOADER - RAVDESS")
print("="*80)

# Verifica percorso
if not ravdess_path or not Path(ravdess_path).exists():
    raise ValueError(f"❌ Dataset RAVDESS non trovato in: {ravdess_path}")

print(f"✅ Usando dataset da: {ravdess_path}\n")

# Il parametro split è rimosso, mostriamo un sample per prova
all_files, speaker_ids = CustomRAVDESSDataset.get_all_speakers(ravdess_path)
print(f"Trovati {len(all_files)} file validi appartenenti a {len(set(speaker_ids))} speaker.")

# Crea il dataset globale o un sottoinsieme per check (is_train applica augmentation)
train_dataset = CustomRAVDESSDataset(dataset_root=ravdess_path, is_train=True)
val_dataset = CustomRAVDESSDataset(dataset_root=ravdess_path, is_train=False)

# Crea i dataloader
batch_size = 32
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Riepilogo dataloader
print("\n" + "="*80)
print("📦 DATALOADER SUMMARY")
print("="*80)
print(f"Train Dataloader:      {len(train_dataloader)} batch × {batch_size} samples = {len(train_dataset)} totali")
print(f"Validation Dataloader: {len(val_dataloader)} batch × {batch_size} samples = {len(val_dataset)} totali")
print("="*80)

🔄 CREAZIONE DATASET E DATALOADER - RAVDESS
✅ Usando dataset da: d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25\data\ravdess

Trovati 1728 file validi appartenenti a 24 speaker.
📊 Statistiche del dataset RAVDESS:

📊 ANALISI RAVDESS TRAINING SET
🔹 Samples Totali: 1728
🔹 Attori (24): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
   - Maschi:  12
   - Femmine: 12

🎭 Distribuzione Emozioni:
   - Angry     :  384 ( 22.2%) ████
   - Happy     :  384 ( 22.2%) ████
   - Neutral   :  576 ( 33.3%) ██████
   - Sad       :  384 ( 22.2%) ████
----------------------------------------
📊 Statistiche del dataset RAVDESS:

📊 ANALISI RAVDESS EVALUATION SET
🔹 Samples Totali: 1728
🔹 Attori (24): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
   - Maschi:  12
   - Femmine: 12

🎭 Distribuzione Emozioni:
   - Angry     :  384 ( 22.2%) ████
   - Happy     :  384 ( 22.2%) ████
   - Neutral   :  576 ( 33.3%) ██████
   - 

In [4]:
# Ricarica il modulo per usare la versione fixata
import importlib
import sys
if 'utils.download_dataset_local' in sys.modules:
    importlib.reload(sys.modules['utils.download_dataset_local'])

from utils.download_dataset_local import dowload_iemocap_local

iemocap_dataset_path = dowload_iemocap_local()
if iemocap_dataset_path:
    print(f"✅ Downloaded IEMOCAP dataset locally in {iemocap_dataset_path}...")
else:
    print("❌ IEMOCAP dataset download failed.")
    
iemocap_path = iemocap_dataset_path


--- Download IEMOCAP (locale) ---
✓ IEMOCAP già presente: d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25\data\iemocap
Numero di file: 81249
✅ Downloaded IEMOCAP dataset locally in d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25\data\iemocap\IEMOCAP_full_release...


In [5]:
# DEBUG: Verifica percorsi IEMOCAP
print("="*80)
print("🔍 DEBUG - VERIFICA PERCORSI IEMOCAP")
print("="*80)

iemocap_debug_path = iemocap_path
print(f"1️⃣  Percorso passato: {iemocap_debug_path}\n")

# Controlla se il percorso esiste
print(f"2️⃣  Percorso esiste: {Path(iemocap_debug_path).exists()}\n")

# Lista cosa c'è dentro
if Path(iemocap_debug_path).exists():
    print(f"3️⃣  Contenuto di {iemocap_debug_path}:")
    for item in Path(iemocap_debug_path).iterdir():
        print(f"   - {item.name} {'(DIR)' if item.is_dir() else ''}")
    print()

# Cerca le cartelle Session
print(f"4️⃣  Ricerca cartelle Session:")
session_folders = list(Path(iemocap_debug_path).glob("Session*"))
print(f"   Trovate: {len(session_folders)} cartelle Session")
for s in session_folders[:3]:
    print(f"   - {s.name}")
print()

# Se ci sono Session, controlla la struttura di una
if session_folders:
    session1 = session_folders[0]
    print(f"5️⃣  Dentro {session1.name}:")
    for item in (session1).iterdir():
        print(f"   - {item.name}")
    print()
    
    # Controlla wav folder
    wav_path = session1 / "sentences" / "wav"
    print(f"6️⃣  Percorso wav: {wav_path}")
    print(f"   Esiste: {wav_path.exists()}")
    if wav_path.exists():
        wav_items = list(wav_path.iterdir())
        print(f"   Contiene {len(wav_items)} elementi:")
        for item in wav_items[:5]:
            print(f"      - {item.name} {'(DIR)' if item.is_dir() else ''}")
    print()
    
    # Controlla label folder
    label_path = session1 / "dialog" / "EmoEvaluation"
    print(f"7️⃣  Percorso label: {label_path}")
    print(f"   Esiste: {label_path.exists()}")
    if label_path.exists():
        label_items = list(label_path.glob("*.txt"))
        print(f"   Trovati {len(label_items)} file .txt")
        for item in label_items[:3]:
            print(f"      - {item.name}")

print("="*80)

🔍 DEBUG - VERIFICA PERCORSI IEMOCAP
1️⃣  Percorso passato: d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25\data\iemocap\IEMOCAP_full_release

2️⃣  Percorso esiste: True

3️⃣  Contenuto di d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25\data\iemocap\IEMOCAP_full_release:
   - Documentation (DIR)
   - Session1 (DIR)
   - Session2 (DIR)
   - Session3 (DIR)
   - Session4 (DIR)
   - Session5 (DIR)

4️⃣  Ricerca cartelle Session:
   Trovate: 5 cartelle Session
   - Session1
   - Session2
   - Session3

5️⃣  Dentro Session1:
   - dialog
   - sentences

6️⃣  Percorso wav: d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25\data\iemocap\IEMOCAP_full_release\Session1\sentences\wav
   Esiste: True
   Contiene 28 elementi:
      - Ses01F_impro01 (DIR)
      - Ses01F_impro02 (DIR)
      - Ses01F_impro03 (DIR)
      - Ses01F_impro04 (DIR)
      - Ses01F_impro05 (DIR)

7️⃣  Percorso label: d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25\data\iemocap\IEMOCAP_full_re

In [6]:
from dataset.custom_iemocap_dataset import CustomIEMOCAPDataset
from utils.get_dataset_statistics import print_iemocap_stats
print("="*80)
print("🔄 CREAZIONE DATASET E DATALOADER - IEMOCAP")
print("="*80)

# Verifica percorso
if not iemocap_path or not Path(iemocap_path).exists():
    raise ValueError(f"❌ Dataset IEMOCAP non trovato in: {iemocap_path}")

print(f"✅ Usando dataset da: {iemocap_path}\n")

# Il parametro split è rimosso
all_files_iemocap, speaker_ids_iemocap = CustomIEMOCAPDataset.get_all_speakers(iemocap_path)
print(f"Trovati {len(all_files_iemocap)} file validi appartenenti a {len(set(speaker_ids_iemocap))} speaker.")

# Crea i dataset (testiamo in blocco)
train_iemocap_dataset = CustomIEMOCAPDataset(dataset_root=iemocap_path, is_train=True)
val_iemocap_dataset = CustomIEMOCAPDataset(dataset_root=iemocap_path, is_train=False)

# Crea i dataloader
batch_size = 32
train_iemocap_dataloader = DataLoader(train_iemocap_dataset, batch_size=batch_size, shuffle=True)
val_iemocap_dataloader = DataLoader(val_iemocap_dataset, batch_size=batch_size, shuffle=False)

# Riepilogo dataloader
print("\n" + "="*80)
print("📦 DATALOADER SUMMARY - IEMOCAP")
print("="*80)
print(f"Train Dataloader:      {len(train_iemocap_dataloader)} batch × {batch_size} samples = {len(train_iemocap_dataset)} totali")
print(f"Validation Dataloader: {len(val_iemocap_dataloader)} batch × {batch_size} samples = {len(val_iemocap_dataset)} totali")
print("="*80)

🔄 CREAZIONE DATASET E DATALOADER - IEMOCAP
✅ Usando dataset da: d:\Roba da D\Poli\ML Vision\speech-emotion-recognition-25\data\iemocap\IEMOCAP_full_release

Trovati 4784 file validi appartenenti a 10 speaker.
✅ Caricate 5531 etichette
🔍 Raccogliendo campioni audio...
✅ Raccolti 2943 campioni audio validi
   - Solo campioni improvvisati
   - Emozioni: ['neutral', 'happy', 'sad', 'angry', 'happy']
📊 Statistiche del dataset IEMOCAP:

📊 ANALISI IEMOCAP TRAINING SET

🔹 SAMPLES TOTALI: 2943
🔹 SESSIONI: ['1', '2', '3', '4', '5']
🔹 SPEAKER UNICI (session, gender): 10
   Elenco: [('1', 'F'), ('1', 'M'), ('2', 'F'), ('2', 'M'), ('3', 'F'), ('3', 'M'), ('4', 'F'), ('4', 'M'), ('5', 'F'), ('5', 'M')]
🔹 IMPROVVISAZIONI UNICHE: 12
   Elenco: ['01', '02', '03', '04', '05', '05a', '05b', '06', '07', '08', '08a', '08b']

👥 SPEAKER INDEPENDENCE (per verificare leakage):
   - Sessione 1: (Ses1, F), (Ses1, M)
   - Sessione 2: (Ses2, F), (Ses2, M)
   - Sessione 3: (Ses3, F), (Ses3, M)
   - Sessione 4: (Ses

 Weights & Biases : Genera i grafici e compara gli esperimenti

In [7]:
import wandb
import os
os.environ['WANDB_API_KEY'] = '7ade30086de7899bed412e3eb5c2da065c146f90'
wandb.login()

wandb: Currently logged in as: pagliarellomatteo (pagliarellomatteo-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
import subprocess
import sys

# Esegui train.py dal project root
result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'ravdess_train' / 'train.py'), '--model', 'CRNN_BiLSTM', '--dataset_path', str(ravdess_path)],
    cwd=str(PROJECT_ROOT),
    capture_output=False
)

#result = subprocess.run(
#    [sys.executable, str(PROJECT_ROOT / 'ravdess_train' / 'train.py'), '--model', 'CRNN_BiGRU'],
#    cwd=str(PROJECT_ROOT)
#)

# Step 5: Evaluate your model



In [9]:
import subprocess
import sys

# Esegui eval.py dal project root
checkpoint_path = PROJECT_ROOT / 'checkpoints' / 'ravdess' / 'best_model.pth'
result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'ravdess_train' / 'eval.py'), '--model', 'CRNN_BiLSTM', '--checkpoint', str(checkpoint_path)],
    cwd=str(PROJECT_ROOT),
    capture_output=False
)

#result = subprocess.run(
#    [sys.executable, str(PROJECT_ROOT / 'ravdess_train' / 'eval.py'), '--model', 'CRNN_BiGRU', '--checkpoint', str(checkpoint_path)],
#    cwd=str(PROJECT_ROOT)
#)